# Lab 03 (cluster) - Thinking vs non-thinking, and why outputs vary

**Chapter 2 (Understanding Foundation Models): Post-Training, Sampling, Test Time Compute, The Probabilistic Nature of AI** | Environment: **cluster** (your Docker JupyterLab) | Time: about 2 hours (do the Colab notebook first)

In the Colab notebook you looked inside the sampling step. Here you work with the course's main model, `qwen-moe`, the way an application does: through an API. `qwen-moe` is a "thinking" model. Post-training taught it to write out a reasoning trace before answering. Chapter 2's **"Test Time Compute"** section is about spending more compute at inference to get better answers, mainly by generating several outputs and picking the best one (best of N, verifiers, majority vote). A reasoning trace is another way to spend inference compute; Chapter 5 ("Give the Model Time to Think") covers its prompting cousin, chain-of-thought. Your job at Larkspur: decide whether thinking is worth its cost for questions technicians actually ask, such as *"A P2 ticket for an Essentials client came in Friday at 4:30 PM. When is the response due?"* You will also test whether `temperature=0` really gives the same answer every time.

**Before you start**
- KU VPN on, `docker compose up`, open this notebook from `labs/lab03-sampling/`.
- The thinking runs in Part B are slow (often 20-90 seconds each). Start Part B when you have 20 minutes to let it run.
- This notebook sends about 30 requests.

## Words you'll meet in this lab

You don't need to memorize these. Each one is explained again where it first matters; come back here if a word stops you.

| Term | What it means | Why it matters |
|---|---|---|
| **Thinking (reasoning) model** | A model post-trained to write out its reasoning (a **reasoning trace**) before giving the answer. `qwen-moe` can do this or skip it. | Reasoning can make answers more accurate, but costs time and tokens. |
| **Test time compute** | Spending extra computation when the model answers (at "test time", as opposed to training) to get a better answer. | Chapter 2's name for the idea this lab tests. |
| **Chain-of-thought** | Asking a model in the prompt to work step by step before answering (Chapter 5). | The prompting version of a thinking model. |
| **Prompt / completion tokens** | Prompt tokens are what you send; completion tokens are what the model writes back, including any reasoning. | Hosted APIs bill completion tokens, and they decide how long you wait. |
| **Latency** | Seconds from sending a request to getting the full reply (from Lab 02). | Technicians feel it directly. |
| **Temperature, greedy** | Sampling settings from the Colab notebook. Temperature 0 means greedy: always take the most likely token. | A.3 and B.4 test whether temperature 0 really gives identical answers. |
| **Deterministic** | Always giving exactly the same output for the same input. | An SLA assistant that changes its answer is hard to trust. |
| **Regular expression (regex)** | A small pattern language for finding text, like "the word ANSWER, then a date". Python's `re` module uses it. | It's how you pull the answer out of a long reply automatically. |
| **SLA, tier, priority** | Larkspur's service level agreement (from Lab 01) sets response targets by the client's **tier** (Essentials, Professional, or Premier: the support plan they pay for) and the ticket's **priority** (P1 most urgent to P4). | Part B's questions are "when is this ticket due?" |
| **Accuracy** | The fraction of answers that are correct. | The main score for comparing the two modes. |
| **Median** | The middle value of a list (from Lab 02). One very slow request can't drag it off the way it can an average. | Used for latency. |
| **Load balancing** | Spreading requests across several machines (from Lab 02). `qwen-moe` runs on two Mac Studios. | One reason outputs can vary between identical requests (Q6). |

## Setup

In [ ]:
import re, time, json
import pandas as pd
import matplotlib.pyplot as plt

from itec632 import get_client, MODELS, NO_THINK, strip_think
from itec632.corpus import load_docs
from itec632.grading import LabChecker

ck = LabChecker("lab03", part="cluster")
client = get_client()
MODEL = MODELS["main"]          # qwen-moe
THINK = {"chat_template_kwargs": {"enable_thinking": True}}

available = {m.id for m in client.models.list().data}
ck.check("setup_model_visible", MODEL in available, 1,
         "Connect to the KU VPN and re-run. If it stays red, run notebooks/00_setup_check.ipynb.");

The helper below sends one request and returns everything we need. Thinking models return their reasoning in one of two ways: a separate `reasoning_content` field on the message (what the course gateway normally does), or a `<think>...</think>` block inside the answer text. The helper handles both, using `strip_think` from the course package for the second case. It also returns `finish_reason`, which says why the model stopped: `stop` means it finished its answer, `length` means it hit the `max_tokens` cap. Just run it.

In [ ]:
def get_reasoning(msg):
    """Reasoning text from a chat message, if the server returned it as a separate field."""
    for key in ("reasoning_content", "reasoning"):
        val = getattr(msg, key, None)
        if not val and getattr(msg, "model_extra", None):
            val = msg.model_extra.get(key)
        if isinstance(val, str) and val.strip():
            return val
    return ""


def ask(question, think, system=None, temperature=None, top_p=None, max_tokens=None):
    """One chat request to qwen-moe. Returns a dict with answer, reasoning, tokens and latency."""
    messages = ([{"role": "system", "content": system}] if system else []) + [{"role": "user", "content": question}]
    kwargs = dict(model=MODEL, messages=messages, extra_body=THINK if think else NO_THINK,
                  max_tokens=max_tokens or (8192 if think else 800))
    if temperature is not None:
        kwargs["temperature"] = temperature
    if top_p is not None:
        kwargs["top_p"] = top_p
    t0 = time.perf_counter()
    r = client.chat.completions.create(**kwargs)
    latency = time.perf_counter() - t0
    msg = r.choices[0].message
    content = msg.content or ""
    reasoning = get_reasoning(msg)
    if not reasoning and "<think>" in content:           # reasoning left inside the text
        m = re.search(r"<think>(.*?)</think>", content, flags=re.S)
        reasoning = m.group(1).strip() if m else ""
        content = strip_think(content)
    return {"answer": content.strip(), "reasoning": reasoning, "latency_s": latency,
            "completion_tokens": r.usage.completion_tokens, "prompt_tokens": r.usage.prompt_tokens,
            "finish_reason": r.choices[0].finish_reason}

# Part A - Guided practice

## A.1 Your first thinking request

Same question, two modes. For each run the cell prints the latency, the number of completion tokens, the answer, and the start of the reasoning (`(none)` if there was none). Watch three things: whether there is reasoning text, how many **completion tokens** you pay for (reasoning tokens count as output tokens), and the latency.

**TODO A.1 (before you run the cell):** guess how many times more completion tokens the thinking run will use than the non-thinking run, and write it in `predicted_token_ratio` at the top of the cell. Any honest guess earns the point.

In [ ]:
predicted_token_ratio = None   # TODO (before you run this cell): e.g. 3 means "thinking uses 3x the tokens"

WARMUP_Q = ("Business hours are Monday-Friday 7:00 AM to 6:00 PM. A ticket with a 4 business hour response "
            "target arrives Monday at 5:00 PM. When is the response due? Answer in one short sentence.")

fast = ask(WARMUP_Q, think=False)
slow = ask(WARMUP_Q, think=True)
for name, r in [("NON-THINKING", fast), ("THINKING", slow)]:
    print(f"=== {name}: {r['latency_s']:.1f}s, {r['completion_tokens']} completion tokens")
    print("answer:", r["answer"][:300])
    print("reasoning (first 600 chars):", (r["reasoning"][:600] or "(none)"), "\n")

The next cell counts the words in the thinking run's reasoning and computes how many times more completion tokens thinking used (both provided). How close was your guess?

In [ ]:
# Provided: words in the reasoning, and thinking completion tokens / non-thinking completion tokens
reasoning_words = len(slow["reasoning"].split())
token_ratio = slow["completion_tokens"] / fast["completion_tokens"] if fast["completion_tokens"] else None
print("reasoning words:", reasoning_words, "| token ratio:", token_ratio, "| your guess:", predicted_token_ratio)

In [ ]:
ck.check("A1_thinking_has_reasoning", bool(slow["reasoning"]) and not fast["reasoning"], 2,
         "The thinking run should return reasoning and the non-thinking run should not. "
         "Check that you passed think=True / think=False.")
ck.check("A1_prediction", isinstance(predicted_token_ratio, (int, float)) and not isinstance(predicted_token_ratio, bool)
         and predicted_token_ratio > 0, 1,
         "Write your guess in predicted_token_ratio at the top of the A.1 request cell (a number). Any honest guess earns the point.")
if token_ratio is not None:
    ck.record("A1_token_ratio", float(token_ratio))
    ck.record("A1_reasoning_words", reasoning_words)
if predicted_token_ratio is not None:
    ck.record("A1_predicted_token_ratio", predicted_token_ratio)

## A.2 Grading answers automatically

To compare modes fairly you need to score many answers without reading each one. A common trick (Chapter 2, **"Structured Outputs"**) is to ask the model to end with a fixed, machine-readable line. We will ask for:

`ANSWER: YYYY-MM-DD HH:MM` (24-hour clock)

Models do not always follow the format exactly (extra bold markers, `9:30` instead of `09:30`), so the parser must be forgiving. It uses a **regular expression** (regex), a small pattern language for finding text: `ANSWER_RE` means "the word ANSWER, a colon, then a date and a time". You don't need to write regex yourself; just finish the last line.

**TODO A.2:** finish `extract_answer` so it returns a normalized string like `"2026-11-09 07:30"`, or `None` if no answer line is found.

In [ ]:
ANSWER_RE = re.compile(r"ANSWER:\W*(\d{4}-\d{2}-\d{2})[ T]+(\d{1,2}):(\d{2})", re.I)


def extract_answer(text):
    """Find the LAST 'ANSWER: YYYY-MM-DD HH:MM' in text and return 'YYYY-MM-DD HH:MM' (hour zero-padded)."""
    matches = ANSWER_RE.findall(text or "")
    if not matches:
        return None
    date, hour, minute = matches[-1]
    # TODO: return the date, a space, then the time as HH:MM. Pad the hour to two digits (8 -> 08);
    #       an f-string format spec or str.zfill() can do the padding.
    return None


tests = {"Due Monday.\nANSWER: 2026-11-09 07:30": "2026-11-09 07:30",
         "**ANSWER: 2026-11-30 8:00**": "2026-11-30 08:00",
         "I am not sure.": None}
for t, want in tests.items():
    print(repr(extract_answer(t)), "expected", repr(want))

In [ ]:
ck.check("A2_extract_answer", all(extract_answer(t) == w for t, w in tests.items()), 3,
         "Return the date, a space, the hour padded to two digits, a colon and the minute, so '8:00' becomes '08:00'. "
         "Keep returning None when there is no match.");

## A.3 Does `temperature=0` make the model deterministic?

In the Colab notebook you saw that temperature 0 means "always take the most likely token" (greedy decoding), which *should* make the output identical every time. Chapter 2 (**"The Probabilistic Nature of AI"**, inconsistency) warns that in practice it often is not. Let's test it: three runs at `temperature=0` and three at `temperature=1.0`, same prompt.

**TODO A.3 (before you run the cell):** will the three `temperature=0` answers be word-for-word identical? Write `"yes"` or `"no"` in `predicted_identical_t0` at the top of the cell. Any honest guess earns the point.

In [ ]:
predicted_identical_t0 = ""   # TODO (before you run this cell): "yes" or "no"

OPEN_Q = ("In two sentences, explain to a Larkspur client why the P3 ticket they emailed on Saturday "
          "was not answered until Monday morning.")
runs_t0 = [ask(OPEN_Q, think=False, temperature=0, max_tokens=120)["answer"] for _ in range(3)]
runs_t1 = [ask(OPEN_Q, think=False, temperature=1.0, max_tokens=120)["answer"] for _ in range(3)]
for i, (a, b) in enumerate(zip(runs_t0, runs_t1)):
    print(f"--- run {i}\n  T=0  : {a}\n  T=1.0: {b}")

The next cell counts how many *distinct* answers each temperature produced (1 means all three were identical). Was your prediction right?

If temperature 0 still gave different answers, here is one reason (a general fact about model servers, not something the chapter goes into): servers process several users' requests together in one batch, and the grouping can change the order of tiny floating-point additions. Those rounding differences can occasionally flip which token is "most likely", and after one different token the rest of the answer can drift. Q6 asks about this.

In [ ]:
# Provided: the number of different answers in each list (a set keeps one copy of each)
distinct_t0 = len(set(runs_t0))
distinct_t1 = len(set(runs_t1))
print("distinct at T=0:", distinct_t0, "  distinct at T=1.0:", distinct_t1, "  | your prediction (identical at T=0?):", predicted_identical_t0)

In [ ]:
ck.check("A3_runs_collected", len(runs_t0) == 3 and len(runs_t1) == 3 and all(runs_t0 + runs_t1), 1,
         "Re-run the A.3 request cell; all six answers should be non-empty.")
ck.check("A3_prediction", str(predicted_identical_t0).strip().lower() in ("yes", "no"), 1,
         "Write \"yes\" or \"no\" in predicted_identical_t0 at the top of the A.3 request cell. Any honest guess earns the point.")
ck.record("A3_distinct_t0", distinct_t0)
ck.record("A3_distinct_t1", distinct_t1)
ck.record("A3_predicted_identical_t0", str(predicted_identical_t0))

# Part B - Applied lab: is thinking worth it for SLA questions?

Technicians constantly ask "when is this ticket due?". The answer depends on the client's tier (Essentials, Professional, or Premier: the support plan they pay for), the priority (P1 is most urgent, P4 least), business hours (7:00 AM to 6:00 PM Central), weekends and holidays. That is multi-step arithmetic with one exact answer, the kind of task where Chapter 2 says extra inference compute pays off (for example, solving a math problem several times and keeping the most common answer).

The model gets Larkspur's current SLA exhibit (the service level agreement, loaded from the course corpus) as its system prompt, plus the 2026 holiday dates, and must end with an `ANSWER:` line. The reference answers were worked out by hand from the exhibit.

In [ ]:
sla = next(d for d in load_docs(folders=("client-docs",)) if d["doc_id"] == "SLA-EXH-2026")
CALENDAR_NOTE = ("2026 calendar notes: Thanksgiving is Thursday, November 26, 2026, and the Friday after is "
                 "November 27. Christmas Eve is Thursday, December 24, 2026 (half day: business hours end at "
                 "12:00 PM CT). Christmas Day is Friday, December 25, 2026. All times are Central Time.")
SYSTEM = ("You are Larkspur Ridge IT's help-desk assistant. Use ONLY the SLA document below.\n"
          "When asked when something is due, work it out carefully, then end your reply with one final line "
          "exactly in the form: ANSWER: YYYY-MM-DD HH:MM (24-hour clock, Central Time).\n\n"
          f"{CALENDAR_NOTE}\n\n=== SLA DOCUMENT ({sla['doc_id']}, version {sla['version']}) ===\n{sla['text']}")

QUESTIONS = [
    ("S1", "An Essentials client emails a P2 ticket on Friday, November 6, 2026 at 4:30 PM. When is the first response due?", "2026-11-09 07:30"),
    ("S2", "A Professional client submits a P2 ticket in the portal on Tuesday, November 10, 2026 at 5:45 PM. When is the response due?", "2026-11-11 07:45"),
    ("S3", "A Premier client phones in a P2 issue on Saturday, November 14, 2026 at 11:20 PM. When is the response due?", "2026-11-15 00:20"),
    ("S4", "An Essentials client opens a P3 ticket on Wednesday, November 25, 2026 at 3:00 PM. When is the response due?", "2026-11-30 08:00"),
    ("S5", "An Essentials client phones in a P1 outage on Monday, November 2, 2026 at 5:30 PM. When is the response due?", "2026-11-03 07:30"),
    ("S6", "A Premier client opens a P3 ticket on Thursday, December 24, 2026 at 11:00 AM. When is the response due?", "2026-12-28 08:00"),
    ("S7", "An Essentials client opens a P2 ticket on Monday, November 16, 2026 at 2:00 PM. When is the resolution target?", "2026-11-17 11:00"),
    ("S8", "An Essentials client emails a P2 ticket on Friday, November 27, 2026 at 10:00 AM. When is the response due?", "2026-11-30 09:00"),
]
print(f"System prompt: {len(SYSTEM):,} characters. {len(QUESTIONS)} questions.")

In [ ]:
# Just run: the parts of the SLA exhibit you need to check the model's work
txt = sla["text"]
for line in txt.splitlines():
    if line.startswith("- **Business hours:**") or line.startswith("- **Business hour:**"):
        print(line)
start, end = txt.find("### 3.1"), txt.find("### 3.3")
print()
print(txt[start:end].strip() if start >= 0 and end > start else "(target table not found)")
print()
print(CALENDAR_NOTE)

**Reading the printout:** the first lines define business hours (Monday-Friday, 7:00 AM-6:00 PM CT, no holidays). The table gives the **response** target for each priority (rows) and tier (columns); notes under it say which targets run only in business hours and which run 24x7.

**Worked example (S1).** An Essentials client emails a P2 ticket on Friday, November 6 at 4:30 PM.
1. Table: Essentials, P2 = **2 business hours**.
2. Friday has business hours until 6:00 PM, so 4:30-6:00 PM uses **1.5 hours**. **0.5 hours** are left.
3. Saturday and Sunday are not business days. The clock restarts Monday, November 9 at 7:00 AM.
4. 7:00 AM + 0.5 hours = **Monday 07:30**, so the answer line is `ANSWER: 2026-11-09 07:30`.

**Try one yourself before B.1:** work out S4 by hand (Essentials, P3, Wednesday, November 25 at 3:00 PM; check the calendar note). Then compare with its reference time in `QUESTIONS` and keep your work for reflection item 4. Doing one by hand makes the reasoning traces in B.3 much easier to judge.

## B.1 Run the benchmark

Each question is asked once without thinking and once with thinking (16 requests). Sampling settings follow Qwen's recommendations for each mode. Just run it and let it work; it prints as it goes. Each line shows the question id, whether thinking was on, the latency, the time the model gave (`predicted`, `None` if it wrote no ANSWER line), the correct time (`ref`), and `OK` or `X`. Thinking requests can take a minute or more each.

In [ ]:
SETTINGS = {False: dict(temperature=0.7, top_p=0.8), True: dict(temperature=0.6, top_p=0.95)}
bench = []
for qid, question, ref in QUESTIONS:
    for think in (False, True):
        try:
            r = ask(question, think=think, system=SYSTEM, **SETTINGS[think])
            r.update(error="")
        except Exception as e:
            r = {"answer": "", "reasoning": "", "latency_s": None, "completion_tokens": None,
                 "prompt_tokens": None, "finish_reason": None, "error": str(e)[:200]}
        r.update(qid=qid, think=think, reference=ref, predicted=extract_answer(r["answer"]))
        r["correct"] = r["predicted"] == ref
        bench.append(r)
        lat = f"{r['latency_s']:.1f}s" if r["latency_s"] else "error"
        print(f"{qid} think={think!s:5} {lat:>7}  predicted={r['predicted']}  ref={ref}  {'OK' if r['correct'] else 'X'}")
        time.sleep(0.5)

bench_df = pd.DataFrame(bench)

## B.2 Compare the two modes

The next cell builds `summary` for you: one row per mode (`think` False/True) with the **accuracy** (the mean of `correct`: True counts as 1 and False as 0, so the mean is the fraction correct), the median `latency_s`, and the mean `completion_tokens`, using only rows without errors. `groupby` splits the rows by mode, like a pivot table in Excel. (Thinking accuracy itself is not graded: the model's score is not under your control, so report whatever you got.)

In [ ]:
ok = bench_df[bench_df["error"] == ""]

# Provided: one row per mode with accuracy, median latency and mean completion tokens
summary = None
if len(ok):
    summary = ok.groupby("think").agg(accuracy=("correct", "mean"),
                                      median_latency_s=("latency_s", "median"),
                                      mean_completion_tokens=("completion_tokens", "mean"))
    display(summary.round(2))
    fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
    labels = ["non-thinking" if not t else "thinking" for t in summary.index]
    for ax, col in zip(axes, ["accuracy", "median_latency_s", "mean_completion_tokens"]):
        ax.bar(labels, summary[col], color=["#4C72B0", "#DD8452"][:len(labels)])
        ax.set_title(col)
    plt.tight_layout(); plt.show()
else:
    print("No successful requests yet: re-run B.1.")

**Reading the table and charts:** the row `False` is non-thinking and `True` is thinking. `accuracy` is the fraction of the 8 questions answered correctly, `median_latency_s` the typical seconds per request, and `mean_completion_tokens` the average number of tokens the model wrote (reasoning included). The three bar charts show the same three columns.

**TODO B.2 (decide):** would you turn thinking on for Larkspur's SLA due-time questions? Set `thinking_for_sla` to `True` or `False` and give a one-sentence reason that uses a number from the table. Either choice can be right; Q4 asks you to defend it.

In [ ]:
# TODO: True or False, and a one-sentence reason with a number from the table
thinking_for_sla = None
thinking_why = ""

In [ ]:
both = summary is not None and {False, True} <= set(summary.index)
ck.check("B1_benchmark_complete", len(ok) >= 14 and ok["think"].nunique() == 2, 3,
         "At least 14 of 16 requests should succeed. Check the VPN and re-run B.1 (it is slow; be patient).")
ck.check("B2_decision", thinking_for_sla in (True, False) and len(str(thinking_why).strip()) >= 20, 2,
         "Set thinking_for_sla to True or False (a boolean, not a string) and write a one-sentence reason in thinking_why.")
ck.check("B2_thinking_costs_more_tokens", both and
         summary.loc[True, "mean_completion_tokens"] > summary.loc[False, "mean_completion_tokens"], 1,
         "Thinking should use more completion tokens on average. If it did not, check that B.1 ran both modes without errors.")
ck.check("B2_answers_parsed", len(ok) > 0 and ok["predicted"].notna().mean() >= 0.75, 2,
         "Fewer than 3 in 4 answers had a readable ANSWER line. Look at bench_df[['qid','think','predicted','reference']]: "
         "if 'predicted' is None, print one raw answer to see its format, then check extract_answer.")
if thinking_for_sla in (True, False):
    ck.record("thinking_for_sla", bool(thinking_for_sla))
if both:
    ck.record("acc_non_thinking", float(summary.loc[False, "accuracy"]))
    ck.record("acc_thinking", float(summary.loc[True, "accuracy"]))
    ck.record("median_latency_non_thinking_s", float(summary.loc[False, "median_latency_s"]))
    ck.record("median_latency_thinking_s", float(summary.loc[True, "median_latency_s"]))
    ck.record("mean_tokens_non_thinking", float(summary.loc[False, "mean_completion_tokens"]))
    ck.record("mean_tokens_thinking", float(summary.loc[True, "mean_completion_tokens"]))

## B.3 Read a reasoning trace

The table below shows each question side by side: one row per question id, with the time each mode gave and the correct `reference` time (`None` means no ANSWER line was found). Pick one question where the two modes disagree, or where thinking got it wrong (if they always agree, pick the question you found hardest). Read the reasoning and see where it went right or wrong. You will use this in analysis Q5.

In [ ]:
def pred_for(qid, think):
    return next((r["predicted"] for r in bench if r["qid"] == qid and r["think"] == think), None)

side = pd.DataFrame([{"qid": q, "non-thinking": pred_for(q, False), "thinking": pred_for(q, True), "reference": ref}
                     for q, _, ref in QUESTIONS]).set_index("qid")
display(side)

In [ ]:
# TODO: set this to the question id you want to inspect, e.g. "S4"
chosen_qid = None

if chosen_qid:
    for _, row in bench_df[bench_df["qid"] == chosen_qid].iterrows():
        print(f"===== {chosen_qid} think={row['think']}  predicted={row['predicted']}  reference={row['reference']}")
        if row["think"]:
            print("--- reasoning (first 3000 chars) ---\n", row["reasoning"][:3000])
        print("--- answer ---\n", row["answer"][:1500], "\n")

In [ ]:
ck.check("B3_chosen_question", chosen_qid in {q for q, _, _ in QUESTIONS}, 1,
         'Set chosen_qid to one of "S1" ... "S8".');

## B.4 Consistency on a factual question at `temperature=0`

An SLA assistant that gives a different due time each time you ask is a problem. A.3 tested free text, where small wording changes don't matter much; here the question is whether the **due time itself** changes. Ask the holiday question (S4) three more times without thinking at `temperature=0`.

**TODO B.4 (before you run the cell):** will all three runs give the same due time? Write `"yes"` or `"no"` in `predicted_same_due_time` at the top of the cell. The cell then counts the distinct predicted answers (after `extract_answer`) and the distinct full texts for you.

In [ ]:
predicted_same_due_time = ""   # TODO (before you run this cell): "yes" or "no"

s4_q, s4_ref = QUESTIONS[3][1], QUESTIONS[3][2]
repeat_texts = [ask(s4_q, think=False, system=SYSTEM, temperature=0)["answer"] for _ in range(3)]
repeat_preds = [extract_answer(t) for t in repeat_texts]
print("predicted:", repeat_preds, " reference:", s4_ref)

# Provided: distinct predicted due times and distinct full texts
distinct_preds = len(set(repeat_preds))
distinct_texts = len(set(repeat_texts))
print("distinct predictions:", distinct_preds, " distinct texts:", distinct_texts, " | your prediction (same due time?):", predicted_same_due_time)

In [ ]:
ck.check("B4_prediction", str(predicted_same_due_time).strip().lower() in ("yes", "no"), 1,
         "Write \"yes\" or \"no\" in predicted_same_due_time at the top of the B.4 cell. Any honest guess earns the point.")
ck.record("B4_distinct_preds_t0", distinct_preds)
ck.record("B4_distinct_texts_t0", distinct_texts)
ck.record("B4_correct_of_3", sum(p == s4_ref for p in repeat_preds))
ck.record("B4_predicted_same_due_time", str(predicted_same_due_time))

# Written analysis

Answer in the box under each question (3-6 sentences). Use **your own numbers**, and cite Chapter 2 by section name where asked. Together with Q2 and Q3 in the Colab notebook (Q1 there is optional), these are graded by the instructor (40 points for the lab).

**Q4. Is thinking worth it?** Report accuracy, median latency, and mean completion tokens for both modes from your B.2 table. Connect the result to Chapter 2's **"Test Time Compute"** section: the chapter's main examples generate several outputs and pick one, so how is a thinking trace similar, and how is it different? For Larkspur's assistant, when would you turn thinking on, and when off? Consider that technicians wait for the answer and that output tokens are what limit throughput on the cluster.

*Your answer:*

**Q5. Inside a reasoning trace.** For the question you chose in B.3, quote one or two short lines from the reasoning (or the non-thinking answer) that show where the answer went right or wrong. Was the error a reasoning error (arithmetic, calendar), a reading error (wrong row of the SLA table), or a format error? What does this suggest about trusting a model's *final answer* without checking the reasoning?

*Your answer:*

**Q6. Determinism.** Did `temperature=0` give identical outputs in A.3 and B.4? Report your distinct counts. Chapter 2 (**"The Probabilistic Nature of AI"**) says that even with fixed sampling settings, outputs can still vary. Give at least two reasons that could apply to *this* cluster (hint: `qwen-moe` is load-balanced across two Mac Studios, and many students share them). What would you do in an SLA assistant to get consistent due times? (Think beyond sampling: could the date math be done by code instead?)

*Your answer:*

# AI-use reflection (10 points, covers both Lab 03 notebooks)

1. What did you use AI assistance for in this lab (tools, which steps)?

*Your answer:*

2. Where did the AI help most, and where was it wrong or unhelpful? How did you notice?

*Your answer:*

3. What is one thing you now understand that you could explain without AI help?

*Your answer:*

4. Before running Part B, did you (or your AI assistant) work out any SLA due time by hand? Did the human, the assistant, or the thinking model make the fewest mistakes, and what does that tell you about where to use each?

*Your answer:*

## Submit
Run this last, save, upload the .ipynb (together with your `lab03_colab.ipynb`).

In [ ]:
ck.report()